# `test_retention.py` — Unit Tests for `RetentionAdvisor`

## Purpose

Validates the retention recommendation pipeline — ensuring turnover probabilities are in the
valid range, employees are correctly assigned to risk zones based on probability thresholds,
strategies are returned for all valid zones, and the final report contains all required fields.

---

## Module Under Test

`src.retention.retention_advisor.RetentionAdvisor`

---

## Test Classes at a Glance

| Class | Methods Tested | What It Verifies |
|-------|----------------|-----------------|
| `TestPredictTurnoverProbabilities` | `predict_turnover_probabilities()` | Range [0,1], length match, instance storage |
| `TestCategorizeRiskZones` | `categorize_risk_zones()` | Valid zone labels, all four threshold boundaries |
| `TestSuggestStrategies` | `suggest_strategies()` | Non-empty string for all zones, invalid zone error |
| `TestGenerateRetentionReport` | `generate_retention_report()` | Required columns, length match, probability range |

---

## Fixtures Used

| Fixture | Source |
|---------|--------|
| `preprocessed_data` | `conftest.py` |
| `advisor_fixture` | local — trains a small `RandomForestClassifier` and wraps it in `RetentionAdvisor` |

---

## How to Run

```bash
pytest tests/test_retention.py -v
```


---

## `TestPredictTurnoverProbabilities`

**Purpose:** Tests `predict_turnover_probabilities()` — verifies that probabilities are
valid floats in [0, 1], that the array length matches the test set, and that results are
cached on the instance for downstream use.

### Test Methods

| Method | Verifies |
|--------|----------|
| `test_probabilities_in_range` | All values ≥ 0.0 and ≤ 1.0 |
| `test_probabilities_length_matches_test` | `len(probs) == len(X_test)` |
| `test_probabilities_stored_on_instance` | `advisor.probabilities` is populated and same length |


In [ ]:
import numpy as np
import pandas as pd
import pytest

from src.retention.retention_advisor import RetentionAdvisor, ZONE_ORDER
from src.utils.exceptions import ModelEvaluationError


@pytest.fixture
def advisor_fixture(preprocessed_data):
    X_train, X_test, y_train, y_test = preprocessed_data
    from sklearn.ensemble import RandomForestClassifier
    model = RandomForestClassifier(n_estimators=10, random_state=42)
    model.fit(X_train, y_train)
    return RetentionAdvisor(model, X_test, y_test)


class TestPredictTurnoverProbabilities:
    def test_probabilities_in_range(self, advisor_fixture):
        probs = advisor_fixture.predict_turnover_probabilities()
        assert np.all(probs >= 0.0)
        assert np.all(probs <= 1.0)

    def test_probabilities_length_matches_test(self, preprocessed_data):
        X_train, X_test, y_train, y_test = preprocessed_data
        from sklearn.ensemble import RandomForestClassifier
        model = RandomForestClassifier(n_estimators=10, random_state=42)
        model.fit(X_train, y_train)
        advisor = RetentionAdvisor(model, X_test, y_test)
        probs = advisor.predict_turnover_probabilities()
        assert len(probs) == len(X_test)

    def test_probabilities_stored_on_instance(self, advisor_fixture):
        probs = advisor_fixture.predict_turnover_probabilities()
        assert advisor_fixture.probabilities is not None
        assert len(advisor_fixture.probabilities) == len(probs)


---

## `TestCategorizeRiskZones`

**Purpose:** Tests `categorize_risk_zones()` — verifies the four probability threshold
boundaries (Safe < 0.20, Low-Risk 0.20–0.60, Medium-Risk 0.60–0.90, High-Risk > 0.90)
are applied correctly.

### Test Methods

| Method | Verifies |
|--------|----------|
| `test_all_zones_in_valid_set` | All returned zones are members of `ZONE_ORDER` |
| `test_safe_zone_threshold` | Probs [0.05, 0.10, 0.19] → all `"Safe"` |
| `test_low_risk_zone_threshold` | Probs [0.25, 0.40, 0.59] → all `"Low-Risk"` |
| `test_medium_risk_zone_threshold` | Probs [0.65, 0.75, 0.89] → all `"Medium-Risk"` |
| `test_high_risk_zone_threshold` | Probs [0.91, 0.95, 0.99] → all `"High-Risk"` |


In [ ]:
class TestCategorizeRiskZones:
    def test_all_zones_in_valid_set(self, advisor_fixture):
        probs = advisor_fixture.predict_turnover_probabilities()
        zones = advisor_fixture.categorize_risk_zones(probs)
        valid_zones = set(ZONE_ORDER)
        for zone in zones:
            assert str(zone) in valid_zones

    def test_safe_zone_threshold(self, advisor_fixture):
        low_probs = np.array([0.05, 0.10, 0.19])
        zones = advisor_fixture.categorize_risk_zones(low_probs)
        assert all(z == "Safe" for z in zones)

    def test_low_risk_zone_threshold(self, advisor_fixture):
        probs = np.array([0.25, 0.40, 0.59])
        zones = advisor_fixture.categorize_risk_zones(probs)
        assert all(z == "Low-Risk" for z in zones)

    def test_medium_risk_zone_threshold(self, advisor_fixture):
        probs = np.array([0.65, 0.75, 0.89])
        zones = advisor_fixture.categorize_risk_zones(probs)
        assert all(z == "Medium-Risk" for z in zones)

    def test_high_risk_zone_threshold(self, advisor_fixture):
        high_probs = np.array([0.91, 0.95, 0.99])
        zones = advisor_fixture.categorize_risk_zones(high_probs)
        assert all(z == "High-Risk" for z in zones)


---

## `TestSuggestStrategies`

**Purpose:** Tests `suggest_strategies()` — verifies a meaningful (non-empty) strategy string
is returned for every valid zone, and that an unknown zone raises a `ValueError`.

### Test Methods

| Method | Verifies |
|--------|----------|
| `test_all_zones_return_string` | All 4 zones return a non-empty `str` |
| `test_invalid_zone_raises` | Raises `ValueError` with "Unknown zone" for unrecognised zone |


In [ ]:
class TestSuggestStrategies:
    def test_all_zones_return_string(self, advisor_fixture):
        for zone in ZONE_ORDER:
            strategy = advisor_fixture.suggest_strategies(zone)
            assert isinstance(strategy, str)
            assert len(strategy) > 0

    def test_invalid_zone_raises(self, advisor_fixture):
        with pytest.raises(ValueError, match="Unknown zone"):
            advisor_fixture.suggest_strategies("InvalidZone")


---

## `TestGenerateRetentionReport`

**Purpose:** Tests `generate_retention_report()` — verifies the output DataFrame contains all
required columns, has the same number of rows as the test set, and that probability values
remain in [0, 1].

### Test Methods

| Method | Verifies |
|--------|----------|
| `test_report_has_required_columns` | Contains `employee_index`, `turnover_probability`, `risk_zone`, `actual_left`, `strategy` |
| `test_report_length_matches_test_set` | `len(report) == len(X_test)` |
| `test_probabilities_in_report_range` | `turnover_probability` column values ∈ [0, 1] |


In [ ]:
class TestGenerateRetentionReport:
    def test_report_has_required_columns(self, advisor_fixture):
        report = advisor_fixture.generate_retention_report()
        expected = {
            "employee_index",
            "turnover_probability",
            "risk_zone",
            "actual_left",
            "strategy",
        }
        assert expected.issubset(set(report.columns))

    def test_report_length_matches_test_set(self, preprocessed_data):
        X_train, X_test, y_train, y_test = preprocessed_data
        from sklearn.ensemble import RandomForestClassifier
        model = RandomForestClassifier(n_estimators=10, random_state=42)
        model.fit(X_train, y_train)
        advisor = RetentionAdvisor(model, X_test, y_test)
        report = advisor.generate_retention_report()
        assert len(report) == len(X_test)

    def test_probabilities_in_report_range(self, advisor_fixture):
        report = advisor_fixture.generate_retention_report()
        assert report["turnover_probability"].between(0, 1).all()
